In [3]:
# ============================================================
# IMPORTS
# ============================================================

import datetime
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR


# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

INPUT_CSV = "../EDA/region_temp_extended.csv"
OUTPUT_DIRS = {
    "knn": Path("../Outputs/KNN"),
    "rf": Path("../Outputs/RandomForest"),
    "svr": Path("../Outputs/SVM"),
}
for path in OUTPUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

RANDOM_STATE = 23
N_JOBS = -1
HORIZON = 1
TUNING_END_YEAR = 2010
TUNING_END_DATE = pd.Timestamp(f"{TUNING_END_YEAR}-12-31")

DATE_COL = "date"
REGION_CANDIDATES = ["name", "region_code"]
TARGET_COL = "next_day"

FEATURE_COLS = [
    "dayofyear",
    "pdtn_doy",
    "de_trend_seas",
    "dts_doyavge",
    "dts_doyvar",
    "yday",
    "IIdays_ago",
    "IIIdays_ago",
    "IVdays_ago",
    "Vdays_ago",
    "VIdays_ago",
    "VIIdays_ago",
    "last_year",
    "last_2year",
    "last_3year",
    "last_4year",
    "last_5year",
    "h1_last_year",
    "h1_last_2years",
    "h1_last_3years",
    "h1_last_4years",
    "h1_last_5years",
    "h1_ly_2days",
    "h1_ly_3day",
    "h1_ly_4day",
    "h1_ly_5day",
    "h1_ly_6day",
    "h1_ly_7day",
    "h1_ly_next_1day",
    "h1_ly_next_2days",
    "h1_ly_next_3days",
    "h1_ly_next_4days",
    "h1_ly_next_5days",
    "h1_ly_next_6days",
    "h1_ly_next_7days",
    "diff_1year",
    "diff_2year",
    "diff_3year",
    "diff_4year",
    "diff_5year",
    "diff_yday",
    "diff_2days",
    "diff_3days",
    "diff_4days",
    "diff_5days",
    "diff_6days",
    "diff_7days",
    "last_7_1day_deltas_mean",
    "last_7_1day_deltas_min",
    "last_7_1day_deltas_max",
]

MODEL_SPECS = [
    {
        "label": "K-Nearest Neighbors Regression",
        "short_name": "knn",
        "model_tag": "paper_knn",
        "estimator": KNeighborsRegressor(),
        "pipeline_step": "knn",
        "param_grid": {
            "knn__n_neighbors": [10, 15, 50, 100, 500],
            "knn__weights": ["uniform", "distance"],
            "knn__algorithm": ["auto", "ball_tree", "kd_tree"],
            "knn__leaf_size": [1, 5, 30, 100, 200],
        },
    },
    {
        "label": "SVM_SVR",
        "short_name": "svr",
        "model_tag": "paper_svm",
        "estimator": SVR(),
        "pipeline_step": "svr",
        "param_grid": {
            "svr__kernel": ["rbf", "linear", "poly", "sigmoid"],
            "svr__degree": [1, 2, 3],
            "svr__gamma": ["scale", "auto"],
            "svr__tol": [0.0001, 0.0005],
            "svr__epsilon": [0.1, 0.2, 0.7, 1.0, 10.0],
            "svr__C": [1.0],
        },
    },
    {
        "label": "Random Forest Regression",
        "short_name": "rf",
        "model_tag": "paper_rf",
        "estimator": RandomForestRegressor(random_state=RANDOM_STATE),
        "pipeline_step": "rf",
        "param_grid": {
            "rf__n_estimators": [100, 200, 400, 500, 700],
            "rf__criterion": ["squared_error", "absolute_error"],
            "rf__max_depth": [2, 3, 4, 15, 20],
            "rf__min_samples_split": [2, 5, 10, 50, 100],
            "rf__max_features": [None, "sqrt", "log2", 0.3],
        },
    },
]

OUTER_CV = TimeSeriesSplit(n_splits=5, test_size=365)
INNER_CV = TimeSeriesSplit(n_splits=3)


# ============================================================
# HELPERS
# ============================================================

def find_region_col(df):
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column among {REGION_CANDIDATES}")


def create_next_day_target(group):
    g = group.copy().sort_values(DATE_COL)
    g[TARGET_COL] = g["de_trend_seas"].shift(-1)
    return g


def load_data():
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    region_col = find_region_col(df)
    required = {DATE_COL, region_col, *FEATURE_COLS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df, region_col


def evaluate_fit(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
        "Bias": float(np.mean(y_pred - y_true)),
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def build_pipeline(spec):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),
            (spec["pipeline_step"], spec["estimator"]),
        ]
    )


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df, region_col = load_data()

    feat_df = (
        feat_df.groupby(region_col, group_keys=False)
        .apply(create_next_day_target)
        .reset_index(drop=True)
    )

    for spec in MODEL_SPECS:
        output_dir = OUTPUT_DIRS[spec["short_name"]]
        nested_cv_results = []
        best_params_dict = {}
        best_params_rows = []
        fit_metric_rows = []

        for region in feat_df[region_col].dropna().unique():
            print(f"\n=== {spec["label"]} | REGION {region} ===")

            region_df = (
                feat_df.loc[feat_df[region_col] == region, [DATE_COL, region_col] + FEATURE_COLS + [TARGET_COL]]
                .copy()
                .sort_values(DATE_COL)
            )
            tuning_df = region_df.loc[region_df[DATE_COL] <= TUNING_END_DATE].copy()
            train = tuning_df.dropna(axis=0, how="any").copy()

            print(f"Tuning rows through {TUNING_END_DATE.date()}: {len(train)}")

            if len(train) < 365 * 6:
                print(f"Skipping {region}: too few rows ({len(train)})")
                continue

            X = train[FEATURE_COLS]
            y = train[TARGET_COL]
            fold_results = []

            for fold_idx, (train_idx, test_idx) in enumerate(OUTER_CV.split(X), start=1):
                print(f"Outer Fold {fold_idx}")

                X_outer_train = X.iloc[train_idx]
                X_outer_test = X.iloc[test_idx]
                y_outer_train = y.iloc[train_idx]
                y_outer_test = y.iloc[test_idx]

                pipe = build_pipeline(spec)
                grid_search = GridSearchCV(
                    estimator=pipe,
                    param_grid=spec["param_grid"],
                    cv=INNER_CV,
                    scoring="neg_mean_absolute_percentage_error",
                    refit=True,
                    n_jobs=N_JOBS,
                )
                grid_search.fit(X_outer_train, y_outer_train)

                best_model = grid_search.best_estimator_
                y_pred = best_model.predict(X_outer_test)

                row = {
                    "model": spec["model_tag"],
                    "region": region,
                    "mape": float(mean_absolute_percentage_error(y_outer_test, y_pred)),
                    "r2": float(r2_score(y_outer_test, y_pred)),
                    "horizon": HORIZON,
                    "fold": fold_idx,
                }
                nested_cv_results.append(row)
                fold_results.append(row)

                fold_key = f"{region}_fold{fold_idx}"
                best_params_dict[fold_key] = {
                    "model": spec["model_tag"],
                    "params": grid_search.best_params_,
                }

                fold_best_params = pd.DataFrame(grid_search.best_params_, index=[fold_key])
                fold_best_params["model"] = spec["model_tag"]
                fold_best_params["region"] = region
                fold_best_params["fold"] = fold_idx
                best_params_rows.append(fold_best_params)

            nested_cv_df = pd.DataFrame(fold_results)
            if nested_cv_df.empty:
                continue

            min_row = nested_cv_df.loc[nested_cv_df["mape"].idxmin()]
            best_fold = f"{min_row["region"]}_fold{int(min_row["fold"])}"
            best_params = best_params_dict[best_fold]["params"]

            final_pipe = build_pipeline(spec)
            final_pipe.set_params(**best_params)
            final_pipe.fit(X, y)

            y_fit = final_pipe.predict(X)
            fit_metrics = evaluate_fit(y, y_fit)
            fit_metrics["model"] = spec["model_tag"]
            fit_metrics["region"] = region
            fit_metric_rows.append(fit_metrics)

            with open(output_dir / f"best_mod_{region}_{spec["model_tag"]}.pkl", "wb") as f:
                pickle.dump(final_pipe, f)

        nested_cv_all = pd.DataFrame(nested_cv_results)
        best_params_all = pd.concat(best_params_rows, axis=0) if best_params_rows else pd.DataFrame()
        fit_metrics_all = pd.DataFrame(fit_metric_rows)

        nested_cv_all.to_csv(output_dir / f"{spec["model_tag"]}_nested_cv_{TODAY}.csv", index=False)
        best_params_all.to_csv(output_dir / f"{spec["model_tag"]}_best_params_{TODAY}.csv")
        fit_metrics_all.to_csv(output_dir / f"{spec["model_tag"]}_fit_metrics_{TODAY}.csv", index=False)
        with open(output_dir / f"{spec["model_tag"]}_nested_cv_best_params_{TODAY}.json", "w") as f:
            json.dump(best_params_dict, f, indent=4)

    print("Time taken:", datetime.datetime.now() - START)


main()



=== K-Nearest Neighbors Regression | REGION 11 ===
Tuning rows through 2010-12-31: 13140
Outer Fold 1


KeyboardInterrupt: 

In [4]:
import datetime
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

INPUT_CSV = "../EDA/region_temp_extended.csv"

SOURCE_DIRS = {
    "knn": Path("../Outputs/KNN"),
    "rf": Path("../Outputs/RandomForest"),
    "svm": Path("../Outputs/SVM"),
}

EXPORT_DIRS = {
    "knn": Path("../Outputs/KNN_rolling_exports"),
    "rf": Path("../Outputs/RandomForest_rolling_exports"),
    "svm": Path("../Outputs/SVM_rolling_exports"),
}
for path in EXPORT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

DATE_COL = "date"
REGION_CANDIDATES = ["name", "region_code"]
TARGET_COL = "next_day"

START_CALIB_YEAR = 2010
END_CALIB_YEAR = 2023

FEATURE_COLS = [
    "dayofyear", "pdtn_doy", "de_trend_seas", "dts_doyavge", "dts_doyvar", "yday",
    "IIdays_ago", "IIIdays_ago", "IVdays_ago", "Vdays_ago", "VIdays_ago", "VIIdays_ago",
    "last_year", "last_2year", "last_3year", "last_4year", "last_5year",
    "h1_last_year", "h1_last_2years", "h1_last_3years", "h1_last_4years", "h1_last_5years",
    "h1_ly_2days", "h1_ly_3day", "h1_ly_4day", "h1_ly_5day", "h1_ly_6day", "h1_ly_7day",
    "h1_ly_next_1day", "h1_ly_next_2days", "h1_ly_next_3days", "h1_ly_next_4days",
    "h1_ly_next_5days", "h1_ly_next_6days", "h1_ly_next_7days",
    "diff_1year", "diff_2year", "diff_3year", "diff_4year", "diff_5year",
    "diff_yday", "diff_2days", "diff_3days", "diff_4days", "diff_5days", "diff_6days",
    "diff_7days", "last_7_1day_deltas_mean", "last_7_1day_deltas_min",
    "last_7_1day_deltas_max",
]

MODEL_CONFIG = {
    "knn": {
        "label": "KNN (pure ML)",
        "pipeline_step": "knn",
        "estimator_cls": KNeighborsRegressor,
        "source_filename_patterns": [
            "best_mod_{region}_paper_knn.pkl",
            "best_mod_{region}_K-Nearest_Neighbors_Regression.pkl",
            "best_mod_{region}_KNN.pkl",
            "best_mod_{region}_knn.pkl",
        ],
        "export_filename": "paper_knn_region_{region}_calib_{calib_year}_predict_{prediction_year}.pkl",
        "candidate_param_names": ["n_neighbors", "weights", "algorithm", "leaf_size", "p", "metric"],
    },
    "svm": {
        "label": "SVR (pure ML)",
        "pipeline_step": "svm",
        "estimator_cls": SVR,
        "source_filename_patterns": [
            "best_mod_{region}_paper_svm.pkl",
            "best_mod_{region}_SVM_SVR.pkl",
            "best_mod_{region}_svm.pkl",
            "best_mod_{region}_SVR.pkl",
        ],
        "export_filename": "paper_svm_region_{region}_calib_{calib_year}_predict_{prediction_year}.pkl",
        "candidate_param_names": ["kernel", "degree", "gamma", "tol", "epsilon", "C", "coef0", "shrinking"],
    },
    "rf": {
        "label": "Random Forest (pure ML)",
        "pipeline_step": "rf",
        "estimator_cls": RandomForestRegressor,
        "source_filename_patterns": [
            "best_mod_{region}_paper_rf.pkl",
            "best_mod_{region}_Random_Forest_Regression.pkl",
            "best_mod_{region}_rf.pkl",
            "best_mod_{region}_RandomForest.pkl",
        ],
        "export_filename": "paper_rf_region_{region}_calib_{calib_year}_predict_{prediction_year}.pkl",
        "candidate_param_names": ["n_estimators", "criterion", "max_depth", "min_samples_split", "max_features", "random_state"],
    },
}

def find_region_col(df):
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column among {REGION_CANDIDATES}")

def create_next_day_target(group):
    g = group.copy().sort_values(DATE_COL)
    g[TARGET_COL] = g["de_trend_seas"].shift(-1)
    return g

def load_data():
    df = pd.read_csv(INPUT_CSV)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    region_col = find_region_col(df)

    required = {DATE_COL, region_col, "de_trend_seas", *FEATURE_COLS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df, region_col

def evaluate_fit(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
        "Bias": float(np.mean(y_pred - y_true)),
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

def build_pipeline(model_key):
    cfg = MODEL_CONFIG[model_key]
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer()),
            ("scaler", StandardScaler()),
            (cfg["pipeline_step"], cfg["estimator_cls"]()),
        ]
    )

def find_source_pkl(model_key, region):
    cfg = MODEL_CONFIG[model_key]
    src_dir = SOURCE_DIRS[model_key]
    for pattern in cfg["source_filename_patterns"]:
        p = src_dir / pattern.format(region=region)
        if p.exists():
            return p
    return None

def extract_best_params_from_model_or_payload(obj, model_key):
    cfg = MODEL_CONFIG[model_key]
    step_name = cfg["pipeline_step"]
    estimator_cls = cfg["estimator_cls"]
    valid_names = cfg["candidate_param_names"]

    if isinstance(obj, dict):
        if "best_params" in obj and isinstance(obj["best_params"], dict):
            best_params = obj["best_params"]
            if all(k.startswith(f"{step_name}__") for k in best_params.keys()):
                return best_params
            return {f"{step_name}__{k}": v for k, v in best_params.items() if k in valid_names}
        if "fitted_model" in obj:
            obj = obj["fitted_model"]
        else:
            raise ValueError("Dict payload does not contain 'best_params' or 'fitted_model'.")

    if hasattr(obj, "named_steps"):
        if step_name not in obj.named_steps:
            raise ValueError(f"Pipeline does not contain a '{step_name}' step.")
        model = obj.named_steps[step_name]
    else:
        model = obj

    if not isinstance(model, estimator_cls):
        raise ValueError(f"Loaded object is not a {estimator_cls.__name__} or Pipeline containing one.")

    raw_params = model.get_params()
    return {f"{step_name}__{name}": raw_params[name] for name in valid_names if name in raw_params}

def get_prediction_year_frame(region_df, prediction_year):
    start = pd.Timestamp(f"{prediction_year}-01-01")
    end = pd.Timestamp(f"{prediction_year}-12-31")
    return region_df.loc[(region_df[DATE_COL] >= start) & (region_df[DATE_COL] <= end)].copy()

feat_df, region_col = load_data()
feat_df = (
    feat_df.groupby(region_col, group_keys=False)
    .apply(create_next_day_target)
    .reset_index(drop=True)
)

for model_key in ["knn", "svm", "rf"]:
    cfg = MODEL_CONFIG[model_key]
    export_dir = EXPORT_DIRS[model_key]
    export_rows = []
    exported_models = {}

    regions = feat_df[region_col].dropna().unique()

    for region in regions:
        print(f"\n=== {model_key.upper()} | REGION {region} ===")
        source_pkl = find_source_pkl(model_key, region)
        if source_pkl is None:
            print(f"Skipping {region}: tuned source file not found")
            continue

        with open(source_pkl, "rb") as f:
            loaded_obj = pickle.load(f)

        try:
            best_params = extract_best_params_from_model_or_payload(loaded_obj, model_key)
        except Exception as e:
            print(f"Skipping {region}: could not extract best params -> {e}")
            continue

        region_df = (
            feat_df.loc[
                feat_df[region_col] == region,
                [DATE_COL, region_col] + FEATURE_COLS + [TARGET_COL]
            ]
            .copy()
            .sort_values(DATE_COL)
            .dropna(axis=0, how="any")
        )

        if region_df.empty:
            continue

        exported_models[region] = {}

        for calib_year in range(START_CALIB_YEAR, END_CALIB_YEAR + 1):
            calibration_end = pd.Timestamp(f"{calib_year}-12-31")
            prediction_year = calib_year + 1

            calib_df = region_df.loc[region_df[DATE_COL] <= calibration_end].copy()
            pred_year_df = get_prediction_year_frame(region_df, prediction_year)

            if len(calib_df) < 365 * 6 or pred_year_df.empty:
                continue

            X_calib = calib_df[FEATURE_COLS]
            y_calib = calib_df[TARGET_COL]
            X_pred_year = pred_year_df[FEATURE_COLS]
            y_pred_year_true = pred_year_df[TARGET_COL]

            final_pipe = build_pipeline(model_key)
            final_pipe.set_params(**best_params)
            final_pipe.fit(X_calib, y_calib)

            y_fit = final_pipe.predict(X_calib)
            fit_metrics = evaluate_fit(y_calib, y_fit)

            y_pred_year_hat = final_pipe.predict(X_pred_year)
            pred_year_metrics = evaluate_fit(y_pred_year_true, y_pred_year_hat)

            payload = {
                "model_name": cfg["label"],
                "region": region,
                "calibration_year": calib_year,
                "calibration_end": str(calibration_end.date()),
                "prediction_year": prediction_year,
                "target_col": TARGET_COL,
                "feature_cols": FEATURE_COLS,
                "best_params": best_params,
                "fit_metrics": fit_metrics,
                "prediction_year_metrics": pred_year_metrics,
                "n_calibration_rows": int(len(calib_df)),
                "n_prediction_rows": int(len(pred_year_df)),
                "train_start": str(calib_df[DATE_COL].min().date()),
                "train_end": str(calib_df[DATE_COL].max().date()),
                "prediction_start": str(pred_year_df[DATE_COL].min().date()),
                "prediction_end": str(pred_year_df[DATE_COL].max().date()),
                "source_tuned_file": str(source_pkl),
                "fitted_model": final_pipe,
                "created_at": datetime.datetime.now().isoformat(),
            }

            out_path = export_dir / cfg["export_filename"].format(
                region=region,
                calib_year=calib_year,
                prediction_year=prediction_year,
            )

            with open(out_path, "wb") as f:
                pickle.dump(payload, f)

            exported_models[region][calib_year] = payload
            print(f"Saved {model_key} | region {region} | calib {calib_year} -> predict {prediction_year}")

    print(f"{model_key} done.")

print("Time taken:", datetime.datetime.now() - START)


=== KNN | REGION 11 ===
Saved knn | region 11 | calib 2010 -> predict 2011
Saved knn | region 11 | calib 2011 -> predict 2012
Saved knn | region 11 | calib 2012 -> predict 2013
Saved knn | region 11 | calib 2013 -> predict 2014
Saved knn | region 11 | calib 2014 -> predict 2015
Saved knn | region 11 | calib 2015 -> predict 2016
Saved knn | region 11 | calib 2016 -> predict 2017
Saved knn | region 11 | calib 2017 -> predict 2018
Saved knn | region 11 | calib 2018 -> predict 2019
Saved knn | region 11 | calib 2019 -> predict 2020
Saved knn | region 11 | calib 2020 -> predict 2021
Saved knn | region 11 | calib 2021 -> predict 2022
Saved knn | region 11 | calib 2022 -> predict 2023
Saved knn | region 11 | calib 2023 -> predict 2024

=== KNN | REGION 24 ===
Saved knn | region 24 | calib 2010 -> predict 2011
Saved knn | region 24 | calib 2011 -> predict 2012
Saved knn | region 24 | calib 2012 -> predict 2013
Saved knn | region 24 | calib 2013 -> predict 2014
Saved knn | region 24 | calib 20